# Ground Truth Builder

Interactive notebook for building `evaluation_data/transaction_link_ground_truth.csv`.

**Workflow**: Define receipt-to-bank-statement pairs, view images inline, enter per-receipt annotations using short-key dicts, then validate and write the CSV.

**Key UX**: No pipe separators to manage — enter one dict per receipt and the notebook joins them automatically.

In [ ]:
"""Cell 1: Setup — discover images and load existing ground truth."""

import csv
import re
from pathlib import Path

from IPython.display import Image, display

EVAL_DIR = Path.cwd() / "evaluation_data"
GT_CSV = EVAL_DIR / "transaction_link_ground_truth.csv"

# Discover images
image_files = sorted(
    p.name for p in EVAL_DIR.iterdir() if p.suffix.lower() in (".png", ".jpg", ".jpeg")
)
print(f"Found {len(image_files)} images in {EVAL_DIR.name}/:")
for f in image_files:
    print(f"  {f}")

# Load existing ground truth
existing_rows: list[dict[str, str]] = []
if GT_CSV.exists():
    with GT_CSV.open(newline="") as f:
        existing_rows = list(csv.DictReader(f))
    print(f"\nExisting ground truth: {len(existing_rows)} rows")
    for row in existing_rows:
        print(f"  {row['image_file']} <-> {row['bank_statement_file']}")
else:
    print("\nNo existing ground truth CSV found — will create new.")

In [ ]:
"""Cell 2: Define receipt-to-bank-statement pairs.

Edit the `pairs` list below. Each tuple is (receipt_image, bank_statement_image).
All files must exist in evaluation_data/.
"""

pairs = [
    ("receipt_image.png", "bank_statement.png"),
    # Add more pairs here...
]

# Validate all files exist
for receipt_file, statement_file in pairs:
    for fname in (receipt_file, statement_file):
        fpath = EVAL_DIR / fname
        if not fpath.exists():
            raise FileNotFoundError(
                f"File not found: {fpath}\n"
                f"Available images: {image_files}"
            )

print(f"{len(pairs)} pairs defined, all files exist.")

In [ ]:
"""Cell 3: Display helper and annotation builder."""


def show_pair(receipt_file: str, statement_file: str) -> None:
    """Display receipt and bank statement images inline."""
    print(f"\n{'=' * 60}")
    print(f"Receipt: {receipt_file}")
    print(f"{'=' * 60}")
    display(Image(filename=str(EVAL_DIR / receipt_file)))

    print(f"\n{'=' * 60}")
    print(f"Bank statement: {statement_file}")
    print(f"{'=' * 60}")
    display(Image(filename=str(EVAL_DIR / statement_file)))


def make_entry(
    receipt_file: str,
    statement_file: str,
    receipts: list[dict[str, str]],
) -> dict[str, str]:
    """Build a ground truth CSV row from per-receipt dicts.

    Each receipt dict has 6 short keys:
        r_date, r_desc, r_total, b_date, b_desc, b_debit

    Returns a dict matching the CSV column names with pipe-joined values.
    """
    PIPE = " | "
    return {
        "image_file": receipt_file,
        "bank_statement_file": statement_file,
        "DOCUMENT_TYPE": "TRANSACTION_LINK",
        "RECEIPT_DATE": PIPE.join(r["r_date"] for r in receipts),
        "RECEIPT_DESCRIPTION": PIPE.join(r["r_desc"] for r in receipts),
        "RECEIPT_TOTAL": PIPE.join(r["r_total"] for r in receipts),
        "BANK_TRANSACTION_DATE": PIPE.join(r["b_date"] for r in receipts),
        "BANK_TRANSACTION_DESCRIPTION": PIPE.join(r["b_desc"] for r in receipts),
        "BANK_TRANSACTION_DEBIT": PIPE.join(r["b_debit"] for r in receipts),
    }


print("Helpers defined: show_pair(), make_entry()")

In [ ]:
"""Cell 4: Annotate each pair.

For each pair, view the images and fill in a list of per-receipt dicts.

Short keys:
    r_date  — receipt date (DD/MM/YYYY)
    r_desc  — receipt store name
    r_total — receipt total with $ prefix
    b_date  — bank transaction date (DD/MM/YYYY)
    b_desc  — bank transaction description (as printed)
    b_debit — bank debit amount with $ prefix
"""

# --- Fill in annotations for each pair ---
# Key = receipt filename, value = list of per-receipt dicts
annotations: dict[str, list[dict[str, str]]] = {
    # Example (replace with your actual data):
    # "receipt_image.png": [
    #     {
    #         "r_date": "15/01/2024",
    #         "r_desc": "Office Supplies Plus",
    #         "r_total": "$83.48",
    #         "b_date": "15/01/2024",
    #         "b_desc": "OFFICE SUPPLIES PLU",
    #         "b_debit": "$83.48",
    #     },
    # ],
}

# Display images for each pair to aid annotation
for receipt_file, statement_file in pairs:
    show_pair(receipt_file, statement_file)

In [ ]:
"""Cell 5: Validate annotations and preview CSV rows."""

DATE_RE = re.compile(r"^\d{2}/\d{2}/\d{4}$")
MONEY_RE = re.compile(r"^(?:\$|€|£)?\d+(?:[\s.,]\d+)*$")
REQUIRED_KEYS = {"r_date", "r_desc", "r_total", "b_date", "b_desc", "b_debit"}
NOT_FOUND = "NOT_FOUND"

errors: list[str] = []
entries: list[dict[str, str]] = []

# Check all pairs have annotations
for receipt_file, statement_file in pairs:
    if receipt_file not in annotations:
        errors.append(f"Missing annotation for {receipt_file}")
        continue

    receipts = annotations[receipt_file]
    if not receipts:
        errors.append(f"{receipt_file}: empty receipts list")
        continue

    for i, r in enumerate(receipts, 1):
        prefix = f"{receipt_file} receipt {i}"

        # Check required keys
        missing = REQUIRED_KEYS - set(r.keys())
        if missing:
            errors.append(f"{prefix}: missing keys {missing}")
            continue

        # Validate date format (NOT_FOUND allowed)
        for key in ("r_date", "b_date"):
            if r[key] == NOT_FOUND:
                continue
            if not DATE_RE.match(r[key]):
                errors.append(f"{prefix}: {key}='{r[key]}' — expected DD/MM/YYYY or NOT_FOUND")

        # Validate monetary format (NOT_FOUND allowed)
        for key in ("r_total", "b_debit"):
            if r[key] == NOT_FOUND:
                continue
            if not MONEY_RE.match(r[key]):
                errors.append(f"{prefix}: {key}='{r[key]}' — expected a number (e.g. $83.48, 1234.56, 7 688,74) or NOT_FOUND")

        # Check descriptions are non-empty (NOT_FOUND allowed)
        for key in ("r_desc", "b_desc"):
            if r[key] == NOT_FOUND:
                continue
            if not r[key].strip():
                errors.append(f"{prefix}: {key} is empty")

    # Build entry
    entries.append(make_entry(receipt_file, statement_file, receipts))

# Report
if errors:
    print(f"VALIDATION FAILED — {len(errors)} error(s):\n")
    for err in errors:
        print(f"  {err}")
else:
    print(f"Validation passed — {len(entries)} row(s) ready.\n")
    # Preview table
    header = [
        "image_file", "bank_statement_file", "DOCUMENT_TYPE",
        "RECEIPT_DATE", "RECEIPT_DESCRIPTION", "RECEIPT_TOTAL",
        "BANK_TRANSACTION_DATE", "BANK_TRANSACTION_DESCRIPTION",
        "BANK_TRANSACTION_DEBIT",
    ]
    for entry in entries:
        print(f"--- {entry['image_file']} <-> {entry['bank_statement_file']} ---")
        for col in header[2:]:
            print(f"  {col}: {entry[col]}")
        print()

In [ ]:
"""Cell 6: Write ground truth CSV."""

# Set to False to overwrite existing CSV (DANGEROUS — default is append)
APPEND_MODE = True

if errors:
    print("Cannot write — fix validation errors first (re-run Cell 5).")
elif not entries:
    print("No entries to write.")
else:
    fieldnames = [
        "image_file", "bank_statement_file", "DOCUMENT_TYPE",
        "RECEIPT_DATE", "RECEIPT_DESCRIPTION", "RECEIPT_TOTAL",
        "BANK_TRANSACTION_DATE", "BANK_TRANSACTION_DESCRIPTION",
        "BANK_TRANSACTION_DEBIT",
    ]

    rows_to_write = entries
    if APPEND_MODE and GT_CSV.exists():
        # Reload CSV fresh to avoid stale existing_rows from Cell 1
        with GT_CSV.open(newline="") as f:
            current_rows = list(csv.DictReader(f))
        current_images = {r["image_file"] for r in current_rows}
        new_entries = [e for e in entries if e["image_file"] not in current_images]
        rows_to_write = current_rows + new_entries
        print(f"Append mode: {len(current_rows)} existing + {len(new_entries)} new")
    else:
        print(f"Overwrite mode: writing {len(rows_to_write)} row(s)")

    with GT_CSV.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows_to_write)

    print(f"Written to: {GT_CSV}")
    print(f"  {len(rows_to_write)} rows, {len(fieldnames)} columns")